**Assignment 1**

Ajdin Muhic, Marcel Rieger

**Stochastic Bandits:**


In [2]:
import math
import random
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches 

class Bandits:
    def __init__(self, num_arms, dist_mode, means):
        self.num_arms = num_arms
        
        # Überprüfung des Verteilungsmodus
        if dist_mode != 'bernoulli' and dist_mode != 'gaussian':
            raise Exception('dist_mode must be equal to bernoulli or gaussian')

        self.means = means
        
        # Falls "random" übergeben wurde, Mittelwerte zufällig generieren
        if isinstance(means, str) and means == "random":
            if dist_mode == "gaussian":
                self.means = np.random.normal(0, 1, num_arms)  # Zufällige Mittelwerte für Gaussian-Verteilung
            elif dist_mode == "bernoulli":
                self.means = np.random.uniform(0, 1, num_arms)  # Zufällige Mittelwerte für Bernoulli-Verteilung
        else:
            self.means = means 
            
        # Initialisierung der Arme entsprechend dem Verteilungsmodus
        if dist_mode == 'gaussian':
            self.arms = {}
            for arm in range(num_arms):
                # Arme als Normalverteilungen mit den angegebenen Mittelwerten
                self.arms[arm] = stats.norm(loc=self.means[arm], scale=1)
        elif dist_mode == 'bernoulli':
            self.arms = {}
            for arm in range(num_arms):
                # Arme als Bernoulli-Verteilungen mit den angegebenen Mittelwerten
                self.arms[arm] = stats.bernoulli(p=self.means[arm])

    def pull_arm(self, arm_i):
        # Ziehen eines Zufallswerts für den angegebenen Arm
        reward = self.arms[arm_i].rvs()  # Ziehe Belohnung aus der Verteilung des Arms
        return reward

Bsp.:

In [3]:
gaussian_bandit = Bandits(num_arms=5, means=[0,.5,10,3,.2], dist_mode='gaussian')
bandit2 = Bandits(num_arms = 3, means = "random", dist_mode= "gaussian")
print(gaussian_bandit.pull_arm(2))
print(bandit2.means)

11.993613053874535
[0.56942517 0.19756157 0.52533694]


**ETC Algo:**

In [4]:
class ExploreThenCommit:
    def __init__(self, bandit, m, n):
        """
        :param bandit: Bandits instance containing arms.
        :param m: Number of times to explore each arm.
        :param n: Total number of rounds.
        """
        self.bandit = bandit
        self.m = m
        self.n = n
        self.k = bandit.num_arms  # Number of arms
        self.arm_expectations = np.zeros(self.k)  # Store estimated expectations
        self.arm_counts = np.zeros(self.k)  # Track number of times each arm was played

    def select_arm(self, round_idx):
        """Choose arm for the current round"""
        if round_idx < self.k * self.m:
            return round_idx % self.k  # Explore arms in a round-robin fashion
        return np.argmax(self.arm_expectations)  # Exploit the best arm found

    def run(self):
        """Executes the Explore-Then-Commit algorithm"""
        best_arm_chosen = np.zeros(self.n)
        self.max_Q = np.max(self.bandit.means)  
        self.regret = np.zeros(self.n)
        self.reward_gap = np.zeros(self.n)
        self.norm_regret = np.zeros(self.n)
        for i in range(self.n):
            arm = self.select_arm(i)
            reward = self.bandit.pull_arm(arm)
            if i == 0:
                self.regret[i] = max(0, self.max_Q - reward)       # Startwert
            else:
                self.regret[i] = self.regret[i - 1] + max(0, self.max_Q - reward)
            self.norm_regret[i] = self.regret[i]/ max(- self.max_Q , self.max_Q)
            # Update expectations during exploration phase
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] +=(reward - self.arm_expectations[arm])/self.arm_counts[arm]
            best_arm_chosen[i] = arm == np.argmax(self.bandit.means)
        probability_to_play_best_arm = np.cumsum(best_arm_chosen) / np.arange(1, self.n + 1)
            

        return self.arm_expectations, np.max(self.arm_expectations), self.regret, self.norm_regret, probability_to_play_best_arm,

Bsp.:

In [5]:
num_arms = 3
means = [0.3, 0.6, 0.8]  # True mean rewards for each arm
bandit = Bandits(num_arms=num_arms, dist_mode='gaussian', means=means)

m = 100  # Explore each arm 5 times
n = 1000  # Total rounds

etc_algo = ExploreThenCommit(bandit, m, n)
arm_expectations, best_arm ,regret, norm_regret ,prob_best = etc_algo.run()

print(f"Arm Expectations: {arm_expectations}")
print(f"Best Arm Selected: {best_arm}")
print(f"Regret over time: {regret[:5]}")
print(f"norm Regret over time: {norm_regret[:5]}")
#print(f"prob to play best arm: {prob_best}")

Arm Expectations: [0.18377615 0.64068739 0.81070333]
Best Arm Selected: 0.810703333103331
Regret over time: [1.18146472 1.18146472 1.18146472 1.9389107  3.10228782]
norm Regret over time: [1.47683089 1.47683089 1.47683089 2.42363837 3.87785977]


**Eps greedy Algo:**


In [3]:
class EpsilonGreedy:
    def __init__(self, bandit, n, eps, q_0):
        """
        :param bandit: Bandit-Instanz mit mehreren Armen.
        :param n: Anzahl der Runden.
        :param eps: Entweder ein einzelner Epsilon-Wert (float) oder eine Liste/Array der Länge n.
        :param q_0: Initiale Erwartungswerte.
        """
        if isinstance(eps, (int, float)):  # Falls eps eine Zahl ist
            eps = np.full(n, eps)  # Erzeuge ein Array mit `n` Kopien von `eps`
        elif len(eps) != n:  # Falls eps eine Liste ist, muss sie die richtige Länge haben
            raise ValueError("eps muss entweder eine Zahl oder eine Liste/Array der Länge n sein.")

        self.bandit = bandit
        self.n = n
        self.eps = np.array(eps)  # Umwandlung in NumPy-Array
        self.k = bandit.num_arms  # Anzahl der Arme
        self.arm_expectations = np.ones(self.k) * q_0  # Erwartungswerte initialisieren
        self.arm_counts = np.zeros(self.k)  # Anzahl der Ziehungen pro Arm

    def select_arm(self, i):
        """ Wählt einen Arm basierend auf dem Epsilon-Greedy-Ansatz für Runde i """
        if np.random.rand() < self.eps[i]:  # Nutze `eps[i]` für die aktuelle Runde
            return np.random.choice(self.k)  # Zufälliger Arm (Exploration)
        else:
            return np.argmax(self.arm_expectations)  # Bester Arm (Exploitation)

    def run(self):
        """ Führt das Epsilon-Greedy-Verfahren aus """
        best_arm_chosen = np.zeros(self.n)
        self.max_Q = np.max(self.bandit.means)  
        self.regret = np.zeros(self.n)
        self.norm_regret = np.zeros(self.n)
        for i in range(self.n):
            arm = self.select_arm(i)
            reward = self.bandit.pull_arm(arm)

            if i == 0: 
                self.regret[i] = max( 0, self.max_Q - reward)
            else:
                self.regret[i] = self.regret[i-1] +  max(0, self.max_Q - reward)

            self.norm_regret[i] = self.regret[i]/ max(- self.max_Q , self.max_Q)

            # Erwartungswerte aktualisieren
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] = self.arm_expectations[arm] + (reward - self.arm_expectations[arm]) / self.arm_counts[arm]
            best_arm_chosen[i] = arm == np.argmax(self.bandit.means)
            probability_to_play_best_arm = np.cumsum(best_arm_chosen) / np.arange(1, self.n + 1)


        return self.arm_expectations, np.argmax(self.arm_expectations), self.regret, self.norm_regret, probability_to_play_best_arm

Bsp.:


In [20]:
num_arms = 3
means = [0.3, 0.6, 0.8]  # True mean rewards for each arm
bandit = Bandits(num_arms=num_arms, dist_mode='gaussian', means=means)

#Inputvariabln festlegen 
n = 500
eps = .05
q_0 = 0

etc_algo = EpsilonGreedy(bandit, n, eps, q_0)
arm_expectations, best_arm , regret, norm_regret, _ = etc_algo.run()

print(f"Arm Expectations: {arm_expectations}")
print(f"Best Arm Selected: {best_arm}")
print(f"Regret: {regret[:5]}")
print(f"norm Regret over time: {norm_regret[:5]}")
#print(f"prob to play best arm: {prob_best}")

Arm Expectations: [-0.00481125  0.63403462  0.49403071]
Best Arm Selected: 1
Regret: [0.73009939 0.73009939 1.33225162 1.33225162 1.33225162]
norm Regret over time: [0.91262424 0.91262424 1.66531452 1.66531452 1.66531452]


**UCB Algo:**

In [4]:
class UCB:
    def __init__(self, bandit, n, delta):
        """
        :param bandit: Bandits-Instanz, die die Arme enthält.
        :param n: Gesamtzahl der Runden.
        :param delta: Delta für den Explorationsterm.
        """
        self.bandit = bandit
        self.n = n
        self.num_arms = bandit.num_arms  # Anzahl der Arme
        self.arm_expectations = np.zeros(self.num_arms)  # Geschätzte Erwartungen für jeden Arm
        self.arm_counts = np.zeros(self.num_arms)  # Anzahl der Ziehungen für jeden Arm
        self.delta = delta  # Delta-Wert für die Exploration
    
    def select_arm(self):
        """
        Wähle den Arm mit der höchsten UCB (Upper Confidence Bound).
        """
        ucb_vals = np.zeros(self.num_arms)
        
        # Berechne den UCB-Wert für alle Arme
        for arm in range(self.num_arms):
            if self.arm_counts[arm] == 0:
                # Falls der Arm noch nie gezogen wurde, setze Erwartung auf den ersten Reward
                ucb_vals[arm] = self.arm_expectations[arm]
            else:
                # Berechne den Explorationsterm: sqrt((2 * log(1/δ)) / N_a(t))
                exploration_bonus = np.sqrt(2 * np.log(1 / self.delta) / self.arm_counts[arm])
                ucb_vals[arm] = self.arm_expectations[arm] + exploration_bonus
        
        # Gib den Arm mit dem höchsten UCB zurück
        return np.argmax(ucb_vals)
    
    def run(self):
        """
        Führe den UCB-Algorithmus für n Runden aus und berechne den Regret.
        """
        best_arm_chosen = np.zeros(self.n)
        self.max_Q = np.max(self.bandit.means)  # Beste Erwartungsbelohnung (beste Arm)
        self.regret = np.zeros(self.n)
        self.norm_regret = np.zeros(self.n)

        # Schritt 1: Jeden Arm einmal spielen
        for arm in range(self.num_arms):
            reward = self.bandit.pull_arm(arm)
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] = reward  # Setze Erwartungswert als ersten Reward

            # Berechne den initialen Regret
            self.regret[arm] = max(0, self.max_Q - reward)
            self.norm_regret[arm] = self.regret[arm] / max(self.max_Q, -self.max_Q)

        # Schritt 2: Greedy Auswahl (argmax der Erwartungen) für die restlichen Runden
        for t in range(self.num_arms, self.n):
            arm = self.select_arm()  # Wähle den Arm basierend auf UCB
            reward = self.bandit.pull_arm(arm)

            # Berechne den Regret
            self.regret[t] = self.regret[t - 1] + max(0, self.max_Q - reward)
            self.norm_regret[t] = self.regret[t] / max(self.max_Q, -self.max_Q)

            # Update der Erwartungen für den Arm
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] += (reward - self.arm_expectations[arm]) / self.arm_counts[arm]
            best_arm_chosen[t] = arm == np.argmax(self.bandit.means)
        probability_to_play_best_arm = np.cumsum(best_arm_chosen) / np.arange(1, self.n + 1)


        return self.arm_expectations, np.argmax(self.arm_expectations), self.regret, self.norm_regret, probability_to_play_best_arm 

Bsp.:

In [23]:
num_arms = 3
means = [0.1, 3, 0.8]  # True mean rewards for each arm
bandit = Bandits(num_arms=num_arms, dist_mode='gaussian', means=means)

#Inputvariablen festlegen 
n = 500 # Total rounds
delta = .05

ucb_algo = UCB(bandit, n=n, delta=delta)
arm_expectations, best_arm , regret , norm_regret, p_best = ucb_algo.run()

print(f"Arm Expectations: {arm_expectations}")
print(f"Best Arm Selected: {best_arm}")
print(f"Regret over time: {regret[:5]}")
print(f"norm Regret over time: {norm_regret[:5]}")
#print(f"prob best arm: {p_best}")

Arm Expectations: [0.545882   3.01512264 1.3309077 ]
Best Arm Selected: 1
Regret over time: [2.454118   0.         1.74109224 1.74109224 2.32981735]
norm Regret over time: [0.81803933 0.         0.58036408 0.58036408 0.77660578]


**Boltzmann Algo:**

In [5]:
class Boltzman:
    def __init__(self, bandit, n, inverse_temp, q_0, adaptive=False, c=1.0):
        """
        :param bandit: Bandits instance containing arms.
        :param n: Total number of rounds.
        :param inverse_temp: Initial inverse temperature θ.
        :param q_0: Initial estimate for Q-values.
        :param adaptive: Whether to adapt θ dynamically.
        :param c: Scaling factor for adaptive θ.
        """
        self.bandit = bandit
        self.n = n
        self.num_arms = bandit.num_arms  
        self.arm_expectations = np.ones(self.num_arms) * q_0  
        self.arm_counts = np.zeros(self.num_arms)  
        self.inverse_temp = inverse_temp  
        self.q_0 = q_0
        self.adaptive = adaptive  
        self.c = c  

    def softmax(self):
        exp_values = np.exp((self.arm_expectations - np.max(self.arm_expectations)) * self.inverse_temp) 
        return exp_values / np.sum(exp_values)  

    def select_arm(self):
        softmax_probs = self.softmax()
        return np.random.choice(self.num_arms, p=softmax_probs) 

    def run(self):
        best_arm_chosen = np.zeros(self.n)
        self.max_Q = np.max(self.bandit.means)  
        self.regret = np.zeros(self.n)
        self.norm_regret = np.zeros(self.n)

        for i in range(self.n):
            arm = self.select_arm()
            reward = self.bandit.pull_arm(arm)

            # Update regret
            self.regret[i] = self.regret[i - 1] + max(0, self.max_Q - reward) if i > 0 else max(0, self.max_Q - reward)
            self.norm_regret[i] = self.regret[i] / max(-self.max_Q, self.max_Q)   

            # Update counts and expectations
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] += (reward - self.arm_expectations[arm]) / self.arm_counts[arm]

            # Update θ adaptively
            if self.adaptive:
                self.inverse_temp = self.c * np.sqrt(self.arm_counts[arm])
            best_arm_chosen[i] = arm == np.argmax(self.bandit.means)
        probability_to_play_best_arm = np.cumsum(best_arm_chosen) / np.arange(1, self.n + 1)


        return self.arm_expectations, np.argmax(self.arm_expectations), self.regret, self.norm_regret, probability_to_play_best_arm 

Bsp.:


In [25]:
num_arms = 4
means = [0.4, 1.2, 0.8, 0.5]  # True mean rewards for each arm
bandit = Bandits(num_arms=num_arms, dist_mode='gaussian', means=means)

n = 1000
boltzman1= Boltzman(bandit, n=n , inverse_temp=.9, q_0=100)
arm_expectations1, best_arm1 , regret1, norm_regret1, _ = boltzman1.run()

print(f"Arm Expectations: {arm_expectations1}")
print(f"Best Arm Selected: {best_arm1}")
print(f"Regret over time: {regret1[:5]}")
print(f"norm Regret over time: {norm_regret1[:5]}")

Arm Expectations: [0.33110067 1.29393414 0.752287   0.53641087]
Best Arm Selected: 1
Regret over time: [1.30987942 1.30987942 1.30987942 1.30987942 1.30987942]
norm Regret over time: [1.09156618 1.09156618 1.09156618 1.09156618 1.09156618]


**Policy gradient Algo:**


In [6]:
class PolicyGradient:
    def __init__(self, bandit, n, learning_rate, baseline):
        """
        :param bandit: Bandits instance containing arms.
        :param m: Number of times to explore each arm.
        :param n: Total number of rounds.
        :param alpha: Learning rate
        """
        self.bandit = bandit
        self.n = n
        self.num_arms = bandit.num_arms  # Number of arms
        self.policy_theta = np.zeros(self.num_arms) #policy parameters
        self.baseline = baseline
        self.alpha = learning_rate
        self.arm_expectations = np.zeros(self.num_arms) 
        self.arm_counts = np.zeros(self.num_arms)  
    
    def softmax(self):
        exp_values = np.exp(self.policy_theta - np.max(self.policy_theta)) #max technique to precent overflow
        softmax_weights = exp_values / np.sum(exp_values)  
        return softmax_weights

    def select_arm(self):
        softmax_probs = self.softmax()
        #print(softmax_probs)
        return np.random.choice(self.num_arms, p=softmax_probs) 

    def update(self, arm, reward):
        #update chosen arms policy
        self.policy_theta[arm] += self.alpha * (reward - self.baseline) * (1-self.policy_theta[arm])
        #update all non-chosen arms
        for a in range(self.num_arms):
            if a != arm:
                self.policy_theta[a] -= (self.alpha * (reward - self.baseline) * self.policy_theta[a])

    def run(self):

        self.max_Q = np.max(self.bandit.means)  
        self.regret = np.zeros(self.n)
        best_arm_chosen = np.zeros(self.n)
    
        for i in range(self.n):
            arm = self.select_arm()
            reward = self.bandit.pull_arm(arm)

            if i == 0:
                self.regret[i] = self.max_Q - reward            # Startwert
            else:
                self.regret[i] = self.regret[i - 1] + (self.max_Q - reward)

            self.update(arm, reward)
            # Update expectations during exploration phase
            self.arm_counts[arm] += 1
            self.arm_expectations[arm] +=(reward - self.arm_expectations[arm])/self.arm_counts[arm]
            best_arm_chosen[i] = arm == np.argmax(self.bandit.means)
        probability_to_play_best_arm = np.cumsum(best_arm_chosen) / np.arange(1, self.n + 1)


        return self.arm_expectations, np.argmax(self.arm_expectations), self.policy_theta, self.softmax(), regret, probability_to_play_best_arm

Bsp.:


In [26]:
num_arms = 3
means = [0.3, 0.6, 0.8]  # True mean rewards for each arm
bandit = Bandits(num_arms=num_arms, dist_mode='gaussian', means=means)

n = 500 # Total rounds
learning_rate = 0.1
base_line = 0.7

policy_gradient = PolicyGradient(bandit, n=n, learning_rate=learning_rate, baseline= base_line)
arm_expectations, best_arm, theta_vector, s_max , regret, _ = policy_gradient.run()

print(f"Arm Expectations: {arm_expectations[:5]}")
print(f"Best Arm Selected: {best_arm}")
print(f"policies: {theta_vector}")
print(f"softmax policies: {s_max}")
print(f"Regret over Time: {regret[:5]}")
print(f"prob best max {np.max(prob_best)}")

Arm Expectations: [0.46324946 0.38277622 0.67150922]
Best Arm Selected: 2
policies: [-1.92710325 -4.8024139   4.94659316]
softmax policies: [1.03351589e-03 5.82888663e-05 9.98908195e-01]
Regret over Time: [2.454118   0.         1.74109224 1.74109224 2.32981735]
prob best max 0.8
